In [2]:
import pandas as pd
import numpy as np

data = {
    'date': [
        '2024-01-01','2024-01-01','2024-01-01','2024-01-01',
        '2024-01-02','2024-01-02','2024-01-02','2024-01-02',
        '2024-01-03','2024-01-03','2024-01-03','2024-01-03',
        '2024-01-04','2024-01-04','2024-01-04','2024-01-04',
    ],
    'campaign_id': [
        'C001','C002','C003','C004',
        'C001','C002','C003','C004',
        'C001','C002','C003','C004',
        'C001','C002','C003','C004',
    ],
    'campaign_name': [
        'Spring Sale','Brand Awareness','App Install','Retargeting',
        'Spring Sale','Brand Awareness','App Install','Retargeting',
        'Spring Sale','Brand Awareness','App Install','Retargeting',
        'Spring Sale','Brand Awareness','App Install','Retargeting',
    ],
    'advertiser': [
        'Nike','Nike','Adidas','Adidas',
        'Nike','Nike','Adidas','Adidas',
        'Nike','Nike','Adidas','Adidas',
        'Nike','Nike','Adidas','Adidas',
    ],
    'impressions': [
        120000, 95000,  43000,  76000,
        134000, 88000,  51000,  82000,
        119000, 91000,  49000,  78000,
        141000, 97000,  0,      80000,   # C003 went dark on day 4
    ],
    'clicks': [
        1800, 950,  430,  380,
        2100, 820,  510,  410,
        1750, 890,  490,  390,
        2200, 940,  0,    400,
    ],
    'conversions': [
        90,  20,  86,  19,
        105, 18,  102, 21,
        88,  22,  98,  20,
        110, 19,  0,   22,
    ],
    'spend': [
        5400.00, 2850.00, 1290.00, 1140.00,
        6300.00, 2460.00, 1530.00, 1230.00,
        5250.00, 2670.00, 1470.00, 1170.00,
        6600.00, 2820.00, 0.00,    1200.00,
    ]
}

df = pd.DataFrame(data)
df['date'] = pd.to_datetime(df['date'])
print(df)


         date campaign_id    campaign_name advertiser  impressions  clicks  \
0  2024-01-01        C001      Spring Sale       Nike       120000    1800   
1  2024-01-01        C002  Brand Awareness       Nike        95000     950   
2  2024-01-01        C003      App Install     Adidas        43000     430   
3  2024-01-01        C004      Retargeting     Adidas        76000     380   
4  2024-01-02        C001      Spring Sale       Nike       134000    2100   
5  2024-01-02        C002  Brand Awareness       Nike        88000     820   
6  2024-01-02        C003      App Install     Adidas        51000     510   
7  2024-01-02        C004      Retargeting     Adidas        82000     410   
8  2024-01-03        C001      Spring Sale       Nike       119000    1750   
9  2024-01-03        C002  Brand Awareness       Nike        91000     890   
10 2024-01-03        C003      App Install     Adidas        49000     490   
11 2024-01-03        C004      Retargeting     Adidas        780

In [15]:
# your answer here
agg = df.groupby(['campaign_id', 'campaign_name']).agg(
    total_impressions=('impressions', 'sum'),
    total_clicks=('clicks', 'sum'),
    total_spend=('spend', 'sum')
).reset_index()


agg['ctr'] = agg['total_clicks'] / agg['total_impressions']
agg['cpc'] = agg['total_spend'] / agg['total_clicks']

agg = agg.sort_values('ctr', ascending=False)
print(agg)


  campaign_id    campaign_name  total_impressions  total_clicks  total_spend  \
0        C001      Spring Sale             514000          7850      23550.0   
2        C003      App Install             143000          1430       4290.0   
1        C002  Brand Awareness             371000          3600      10800.0   
3        C004      Retargeting             316000          1580       4740.0   

        ctr  cpc  
0  0.015272  3.0  
2  0.010000  3.0  
1  0.009704  3.0  
3  0.005000  3.0  


In [ ]:
agg_2 = df.groupby(['advertiser','date'])[['spend','campaign_name']].idxmax()

print(agg_2)

# ── WHAT WENT WRONG ──────────────────────────────────────────────────────────
# idxmax() here returns the row INDEX of the max-spend campaign within each
# advertiser+date group — not the max-spend DAY per advertiser.
# You also skipped step 1: summing spend across campaigns before finding the peak day.
# campaign_name is a string — idxmax() on strings = alphabetical max, not useful.

# ── CORRECT APPROACH — two steps ─────────────────────────────────────────────

# Step 1: sum spend across campaigns → one row per advertiser per day
daily = df.groupby(['advertiser', 'date'], as_index=False)['spend'].sum()

# Step 2: for each advertiser, grab the row where spend is highest
# idxmax() here returns the INDEX of the max-spend day within each advertiser group
# .loc[] then pulls those actual rows from daily
result = daily.loc[daily.groupby('advertiser')['spend'].idxmax()]

print(result)

# ── ALTERNATIVE — sort + drop_duplicates (same result, easier to read) ───────
# daily.sort_values('spend', ascending=False).drop_duplicates(subset='advertiser')



#cant we do daily = df.groupby(['advertiser', 'date'], as_index=False)['spend'].sum().idxmax()
#print(df['date'].loc[daily]) - since daily has the index of the highest spend we just use the index to find the date that had the highest spend no?

#no because the idxmax is the max value per column (not by per groupby values)

                       spend  campaign_name
advertiser date                            
Adidas     2024-01-01      2              3
           2024-01-02      6              7
           2024-01-03     10             11
           2024-01-04     15             15
Nike       2024-01-01      0              0
           2024-01-02      4              4
           2024-01-03      8              8
           2024-01-04     12             12
  advertiser       date   spend
1     Adidas 2024-01-02  2760.0
7       Nike 2024-01-04  9420.0


In [33]:
agg

,campaign_id,campaign_name,total_impressions,total_clicks,total_spend,ctr,cpc
0,C001,Spring Sale,514000,7850,23550.0,0.015272,3.0
2,C003,App Install,143000,1430,4290.0,0.010000,3.0
1,C002,Brand Awareness,371000,3600,10800.0,0.009704,3.0
3,C004,Retargeting,316000,1580,4740.0,0.005000,3.0


In [44]:
df['ctr'] = df['clicks'] / df['impressions']
avg_ctr = df['ctr'].mean()          # ← correct: mean of the 16 row-level CTRs
df['low_ctr'] = df['ctr'] < avg_ctr # ← also use < not <=, "below" means strictly less


In [ ]:
#problem #4
sorted_df = df.sort_values(['campaign_id', 'date'])

sorted_df['spend'] = sorted_df['spend'].astype(int)

print(sorted_df['spend'].dtype)

df['spend_change'] = sorted_df.groupby('campaign_id')['spend'].diff(periods = 1)

int64


In [71]:
impression_missing_and_once = df.groupby(['campaign_id','date'])['impressions'].agg(["max","min"])

print(impression_missing_and_once)

# ── WHAT WENT WRONG ───────────────────────────────────────────────────────────
# Grouped by campaign_id + date — but each campaign has exactly one row per date,
# so min and max of a single value are always identical. Nothing to compare.
# You need to group by campaign_id ONLY so min/max span across all days.

# ── CORRECT APPROACH ──────────────────────────────────────────────────────────

agg_5 = df.groupby(['campaign_id', 'campaign_name'])['impressions'].agg(
    min_impressions='min',
    max_impressions='max'
).reset_index()

print(agg_5)

# campaigns where at least one day was 0 AND at least one day was non-zero
result_5 = agg_5[(agg_5['min_impressions'] == 0) & (agg_5['max_impressions'] > 0)]

print(result_5[['campaign_id', 'campaign_name']])
# Expected: C003 (App Install) — went dark on Jan 4 but was active on days 1-3


                           max     min
campaign_id date                      
C001        2024-01-01  120000  120000
            2024-01-02  134000  134000
            2024-01-03  119000  119000
            2024-01-04  141000  141000
C002        2024-01-01   95000   95000
            2024-01-02   88000   88000
            2024-01-03   91000   91000
            2024-01-04   97000   97000
C003        2024-01-01   43000   43000
            2024-01-02   51000   51000
            2024-01-03   49000   49000
            2024-01-04       0       0
C004        2024-01-01   76000   76000
            2024-01-02   82000   82000
            2024-01-03   78000   78000
            2024-01-04   80000   80000
  campaign_id    campaign_name  min_impressions  max_impressions
0        C001      Spring Sale           119000           141000
1        C002  Brand Awareness            88000            97000
2        C003      App Install                0            51000
3        C004      Retargeting        

In [91]:
#problem 6
campaign_total_spend= df.groupby(["advertiser"])['spend'].transform("sum")

print(campaign_total_spend)


print((df["spend"]/campaign_total_spend)*100)

0     34350.0
1     34350.0
2      9030.0
3      9030.0
4     34350.0
5     34350.0
6      9030.0
7      9030.0
8     34350.0
9     34350.0
10     9030.0
11     9030.0
12    34350.0
13    34350.0
14     9030.0
15     9030.0
Name: spend, dtype: float64
0     15.720524
1      8.296943
2     14.285714
3     12.624585
4     18.340611
5      7.161572
6     16.943522
7     13.621262
8     15.283843
9      7.772926
10    16.279070
11    12.956811
12    19.213974
13     8.209607
14     0.000000
15    13.289037
Name: spend, dtype: float64


In [100]:
cpi = df['spend']/df['conversions']
median_cpi = cpi.median()
print(median_cpi)

total_agg_per_camp = df.groupby(["campaign_id","campaign_name"])[["spend","conversions"]].sum()

total_agg_per_camp['CPI']= (total_agg_per_camp['spend']/total_agg_per_camp['conversions'])<median_cpi

print(total_agg_per_camp)

60.0
                               spend  conversions    CPI
campaign_id campaign_name                               
C001        Spring Sale      23550.0          393   True
C002        Brand Awareness  10800.0           79  False
C003        App Install       4290.0          286   True
C004        Retargeting       4740.0           82   True


In [101]:
agg_7 = df.groupby(['campaign_id', 'campaign_name'])[['spend', 'conversions']].sum().reset_index()
agg_7['cpi'] = agg_7['spend'] / agg_7['conversions']

median_cpi = agg_7['cpi'].median()  # median of 4 campaign-level CPIs, not 16 rows

result_7 = agg_7[agg_7['cpi'] < median_cpi][['campaign_id', 'campaign_name', 'cpi']]
print(result_7)

  campaign_id campaign_name        cpi
2        C003   App Install  15.000000
3        C004   Retargeting  57.804878


In [ ]:
# ── PROBLEM 8: 3-day rolling average spend per campaign ──────────────────────

# CONCEPT: rolling() computes a sliding window calculation along rows.
# window=3 means: current row + 2 rows before it.
# Without groupby, the window would bleed across campaigns (last row of C001
# into first row of C002). So you must groupby campaign first.

# Step 1: sort so each campaign's days are in order
df = df.sort_values(['campaign_id', 'date'])

# Step 2: groupby campaign, then apply rolling(3).mean() within each group
# min_periods=1 means: don't produce NaN when fewer than 3 rows exist yet —
# use however many rows are available (day 1 = avg of 1 day, day 2 = avg of 2 days)
df['rolling_spend_3d'] = (
    df.groupby('campaign_id')['spend']
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

print(df[['campaign_id', 'campaign_name', 'date', 'spend', 'rolling_spend_3d']])

# ── HOW rolling() WORKS ───────────────────────────────────────────────────────
# For C001 (Spring Sale):
#   Jan 1: only 1 day available  → avg(5400)              = 5400.00
#   Jan 2: 2 days available      → avg(5400, 6300)         = 5850.00
#   Jan 3: 3 days available      → avg(5400, 6300, 5250)   = 5650.00
#   Jan 4: window slides forward → avg(6300, 5250, 6600)   = 6050.00
#           (Jan 1 drops out — window always looks back 3 days max)

# ── COMMON rolling() OPTIONS ─────────────────────────────────────────────────
# .rolling(3).mean()   → 3-period average
# .rolling(3).sum()    → 3-period sum (running total over window)
# .rolling(3).min/max()→ 3-period min/max
# window=7             → 7-day rolling (common for weekly smoothing in adtech)
# min_periods=1        → allow partial windows at the start instead of NaN
